# Lab 3 Extended: PCA/LDA Pipeline with Regularization, Class Balancing & K-Fold CV

**Objective:** Extend the text-to-latent representation pipeline by adding PCA and LDA for dimensionality reduction, implementing k-fold cross-validation with regularization, applying class balancing, and visualizing at each step.

**Note:** This is a new notebook; the original notebook is not modified.

In [ ]:
import os
import random
import re
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix
from sklearn.utils import resample

SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)

print(f"Python {sys.version.split()[0]} | PyTorch {torch.__version__}")
print(f"NumPy {np.__version__} | pandas {pd.__version__}")

## 1. Dataset Creation

In [ ]:
texts = [
    "The product quality is excellent",
    "I really love this product",
    "The quality of this item is amazing",
    "The product performance is poor",
    "I dislike this product completely",
]
labels = np.array([1, 1, 1, 0, 0], dtype=np.float32)

review_df = pd.DataFrame({
    "review_id": np.arange(1, len(texts) + 1),
    "text": texts,
    "label": labels.astype(int),
    "sentiment": np.where(labels == 1, "positive", "negative"),
})
display(review_df)

## 2. Text Preprocessing

In [ ]:
def normalize_text(text: str) -> str:
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    return re.sub(r"\s+", " ", text).strip()

cleaned_texts = [normalize_text(text) for text in texts]
preprocessing_df = pd.DataFrame({
    "original_text": texts,
    "cleaned_text": cleaned_texts,
})
preprocessing_df["token_count"] = preprocessing_df["cleaned_text"].str.split().str.len()
display(preprocessing_df)

## 3. Vocabulary and Tokenization

In [ ]:
word2idx = {"<PAD>": 0}
idx = 1
for text in cleaned_texts:
    for token in text.split():
        if token not in word2idx:
            word2idx[token] = idx
            idx += 1

vocab_size = len(word2idx)
vocab_table = pd.DataFrame([
    {"token": word, "token_id": tid}
    for word, tid in sorted(word2idx.items(), key=lambda x: x[1])
])
display(vocab_table)

sequences = []
for text in cleaned_texts:
    sequences.append([word2idx[token] for token in text.split()])

sequence_df = pd.DataFrame({
    "review_id": np.arange(1, len(texts) + 1),
    "cleaned_text": cleaned_texts,
    "token_sequence": sequences,
})
display(sequence_df)

## 4. Padding Sequences

In [ ]:
max_length = max(map(len, sequences))
padded_sequences = np.zeros((len(sequences), max_length), dtype=np.int64)
for i, seq in enumerate(sequences):
    padded_sequences[i, :len(seq)] = seq

padded_df = pd.DataFrame(
    padded_sequences,
    index=[f"review_{review_id}" for review_id in review_df["review_id"]],
    columns=[f"position_{i + 1}" for i in range(max_length)],
)
display(padded_df)
print(f"Padded tensor shape: {padded_sequences.shape}")

## 5. Embedding Classifier with Regularization (L2) and Class Balancing

Using PyTorch with weight decay (L2 regularization) and class-weighted loss for class balancing.

In [ ]:
class ReviewEmbeddingClassifier(nn.Module):
    def __init__(self, vocab_size, embedding_dim, dropout_rate=0.2):
        super(ReviewEmbeddingClassifier, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        self.dropout = nn.Dropout(dropout_rate)
        self.fc = nn.Linear(embedding_dim, 1)
        
    def forward(self, x):
        mask = (x != 0).unsqueeze(-1).float()
        emb = self.embedding(x)
        emb = emb * mask
        summed = emb.sum(dim=1)
        counts = mask.sum(dim=1).clamp(min=1)
        pooled = summed / counts
        pooled = self.dropout(pooled)
        logits = self.fc(pooled).squeeze(-1)
        return logits, emb

embedding_dimension = 8
model = ReviewEmbeddingClassifier(vocab_size, embedding_dimension, dropout_rate=0.2)
print(model)

## 6. K-Fold Cross Validation with Regularization and Class Balancing

Using StratifiedKFold. For each fold: apply random oversampling for class balancing, train with L2 regularization (weight decay).

In [ ]:
X_tensor = torch.LongTensor(padded_sequences)
y_tensor = torch.FloatTensor(labels)

skf = StratifiedKFold(n_splits=2, shuffle=True, random_state=SEED)
cv_results = []

for fold, (train_idx, val_idx) in enumerate(skf.split(padded_sequences, labels.astype(int))):
    print(f"\n{'='*40}")
    print(f"Fold {fold + 1}")
    print(f"{'='*40}")
    
    X_train_fold = padded_sequences[train_idx]
    y_train_fold = labels[train_idx]
    X_val_fold = padded_sequences[val_idx]
    y_val_fold = labels[val_idx]
    
    # Class balancing via oversampling
    pos_mask = y_train_fold == 1
    neg_mask = y_train_fold == 0
    X_pos = X_train_fold[pos_mask]
    y_pos = y_train_fold[pos_mask]
    X_neg = X_train_fold[neg_mask]
    y_neg = y_train_fold[neg_mask]
    
    if len(X_pos) < len(X_neg):
        X_pos_os, y_pos_os = resample(X_pos, y_pos, replace=True, n_samples=len(X_neg), random_state=SEED)
        X_train_bal = np.vstack([X_neg, X_pos_os])
        y_train_bal = np.concatenate([y_neg, y_pos_os])
    elif len(X_neg) < len(X_pos):
        X_neg_os, y_neg_os = resample(X_neg, y_neg, replace=True, n_samples=len(X_pos), random_state=SEED)
        X_train_bal = np.vstack([X_pos, X_neg_os])
        y_train_bal = np.concatenate([y_pos, y_neg_os])
    else:
        X_train_bal = X_train_fold
        y_train_bal = y_train_fold
    
    print(f"Original train: {len(X_train_fold)} (pos={np.sum(y_train_fold)}, neg={len(y_train_fold)-np.sum(y_train_fold)})")
    print(f"Balanced train: {len(X_train_bal)} (pos={np.sum(y_train_bal)}, neg={len(y_train_bal)-np.sum(y_train_bal)})")
    
    # Train model for this fold
    fold_model = ReviewEmbeddingClassifier(vocab_size, embedding_dimension, dropout_rate=0.2)
    criterion = nn.BCEWithLogitsLoss()
    optimizer = optim.Adam(fold_model.parameters(), lr=0.03, weight_decay=0.01)  # L2 regularization
    
    X_train_b = torch.LongTensor(X_train_bal)
    y_train_b = torch.FloatTensor(y_train_bal)
    
    fold_model.train()
    for epoch in range(100):
        optimizer.zero_grad()
        logits, _ = fold_model(X_train_b)
        loss = criterion(logits, y_train_b)
        loss.backward()
        optimizer.step()
    
    # Evaluate
    fold_model.eval()
    with torch.no_grad():
        X_val_b = torch.LongTensor(X_val_fold)
        logits_val, _ = fold_model(X_val_b)
        preds_val = torch.sigmoid(logits_val)
        preds_val_np = (preds_val >= 0.5).float().numpy()
    
    acc = accuracy_score(y_val_fold, preds_val_np)
    prec, rec, f1, _ = precision_recall_fscore_support(y_val_fold, preds_val_np, average='binary', zero_division=0)
    print(f"Validation - Acc: {acc:.4f}, Prec: {prec:.4f}, Rec: {rec:.4f}, F1: {f1:.4f}")
    cv_results.append({'fold': fold+1, 'acc': acc, 'prec': prec, 'rec': rec, 'f1': f1})

cv_df = pd.DataFrame(cv_results)
print(f"\n{'='*40}")
print("Cross-Validation Summary (Stratified 2-Fold)")
print(f"{'='*40}")
display(cv_df)
print(f"Mean Accuracy: {cv_df['acc'].mean():.4f} ± {cv_df['acc'].std():.4f}")
print(f"Mean F1: {cv_df['f1'].mean():.4f} ± {cv_df['f1'].std():.4f}")

## 7. Final Model Training with Balancing and Regularization

Train on full dataset with class balancing and L2 regularization to extract final embeddings.

In [ ]:
# Class balancing on full dataset
pos_mask_full = labels == 1
neg_mask_full = labels == 0
X_pos_full = padded_sequences[pos_mask_full]
y_pos_full = labels[pos_mask_full]
X_neg_full = padded_sequences[neg_mask_full]
y_neg_full = labels[neg_mask_full]

if len(X_pos_full) < len(X_neg_full):
    X_pos_os, y_pos_os = resample(X_pos_full, y_pos_full, replace=True, n_samples=len(X_neg_full), random_state=SEED)
    X_train_final = np.vstack([X_neg_full, X_pos_os])
    y_train_final = np.concatenate([y_neg_full, y_pos_os])
elif len(X_neg_full) < len(X_pos_full):
    X_neg_os, y_neg_os = resample(X_neg_full, y_neg_full, replace=True, n_samples=len(X_pos_full), random_state=SEED)
    X_train_final = np.vstack([X_pos_full, X_neg_os])
    y_train_final = np.concatenate([y_pos_full, y_neg_os])
else:
    X_train_final = padded_sequences
    y_train_final = labels

final_model = ReviewEmbeddingClassifier(vocab_size, embedding_dimension, dropout_rate=0.2)
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(final_model.parameters(), lr=0.03, weight_decay=0.01)

X_train_f = torch.LongTensor(X_train_final)
y_train_f = torch.FloatTensor(y_train_final)

losses = []
final_model.train()
for epoch in range(150):
    optimizer.zero_grad()
    logits, _ = final_model(X_train_f)
    loss = criterion(logits, y_train_f)
    loss.backward()
    optimizer.step()
    losses.append(loss.item())

plt.figure(figsize=(8, 4))
plt.plot(losses, color='#b23a48')
plt.title('Training Loss (with L2 Regularization & Class Balancing)')
plt.xlabel('Epoch')
plt.ylabel('BCE Loss')
plt.grid(alpha=0.25)
plt.tight_layout()
plt.show()
print(f"Final loss: {losses[-1]:.4f}")

## 8. Extract Learned Embeddings

In [ ]:
final_model.eval()
with torch.no_grad():
    X_all = torch.LongTensor(padded_sequences)
    logits_all, emb_all = final_model(X_all)
    emb_all_np = emb_all.numpy()
    embedding_matrix = final_model.embedding.weight.detach().numpy()

sentence_embeddings = emb_all_np.copy()
padding_mask = padded_sequences[..., np.newaxis] == 0
sentence_embeddings = np.where(padding_mask, 0.0, sentence_embeddings)

print(f"Embedding tensor shape: {emb_all_np.shape}")
print(f"Vocabulary embedding matrix shape: {embedding_matrix.shape}")

words = [word for word, tid in sorted(word2idx.items(), key=lambda x: x[1]) if word != '<PAD>']
word_vectors = pd.DataFrame(
    embedding_matrix[1:],
    index=words,
    columns=[f"dim_{i + 1}" for i in range(embedding_dimension)],
)
word_vectors.index.name = "word"
display(word_vectors.round(4))

## 9. Flatten Sentence Embeddings

In [ ]:
flattened_vectors = sentence_embeddings.reshape(len(texts), -1)
flattened_columns = [
    f"position_{position + 1}_dim_{dimension + 1}"
    for position in range(max_length)
    for dimension in range(embedding_dimension)
]
flattened_df = pd.DataFrame(flattened_vectors, columns=flattened_columns)
flattened_df.insert(0, "review_id", review_df["review_id"].values)
flattened_df.insert(1, "sentiment", review_df["sentiment"].values)
display(flattened_df.round(4))
print(f"Flattened vector shape: {flattened_vectors.shape}")

## 10. PCA Pipeline - Visualization at Each Step

In [ ]:
# PCA on word embeddings
word_vectors_for_pca = embedding_matrix[1:]
pca_words = PCA(n_components=2, random_state=SEED)
pca_words_coords = pca_words.fit_transform(word_vectors_for_pca)
pca_words_df = pd.DataFrame({
    "word": words,
    "PC1": pca_words_coords[:, 0],
    "PC2": pca_words_coords[:, 1],
})
display(pca_words_df.round(4))
print("Explained variance ratio (words):", np.round(pca_words.explained_variance_ratio_, 4))

fig, ax = plt.subplots(figsize=(11, 7))
ax.scatter(pca_words_df["PC1"], pca_words_df["PC2"], s=90, color="#277da1", edgecolor="white", linewidth=0.8)
for _, row in pca_words_df.iterrows():
    ax.annotate(row["word"], (row["PC1"], row["PC2"]), xytext=(6, 6), textcoords="offset points", fontsize=10)
ax.axhline(0, color="gray", linewidth=0.8, alpha=0.4)
ax.axvline(0, color="gray", linewidth=0.8, alpha=0.4)
ax.set(title="PCA: Word Embeddings (Step-wise Visualization)", 
       xlabel=f"PC1 ({pca_words.explained_variance_ratio_[0]:.1%} variance)",
       ylabel=f"PC2 ({pca_words.explained_variance_ratio_[1]:.1%} variance)")
ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()

In [ ]:
# PCA on flattened sentence embeddings
pca_sent = PCA(n_components=2, random_state=SEED)
pca_sent_coords = pca_sent.fit_transform(flattened_vectors)
pca_sent_df = pd.DataFrame({
    "review_id": review_df["review_id"],
    "sentiment": review_df["sentiment"],
    "PC1": pca_sent_coords[:, 0],
    "PC2": pca_sent_coords[:, 1],
})
display(pca_sent_df.round(4))
print("Explained variance ratio (sentences):", np.round(pca_sent.explained_variance_ratio_, 4))

fig, ax = plt.subplots(figsize=(10, 7))
colors = {'positive': '#2a9d8f', 'negative': '#e76f51'}
for sent, group in pca_sent_df.groupby('sentiment'):
    ax.scatter(group['PC1'], group['PC2'], s=150, label=sent, color=colors[sent], edgecolor='black', linewidth=0.8)
    for _, row in group.iterrows():
        ax.annotate(f"R{row['review_id']}", (row['PC1'], row['PC2']), xytext=(5, 5), textcoords="offset points", fontsize=9)
ax.axhline(0, color="gray", linewidth=0.8, alpha=0.4)
ax.axvline(0, color="gray", linewidth=0.8, alpha=0.4)
ax.set(title="PCA: Flattened Sentence Embeddings (Step-wise Visualization)", 
       xlabel=f"PC1 ({pca_sent.explained_variance_ratio_[0]:.1%} variance)",
       ylabel=f"PC2 ({pca_sent.explained_variance_ratio_[1]:.1%} variance)")
ax.legend()
ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()

## 11. LDA Pipeline - Supervised Dimensionality Reduction with Visualization

In [ ]:
# LDA on word embeddings with pseudo-labels based on sentence occurrence
word_to_label_votes = {w: [] for w in words}
for i, text in enumerate(cleaned_texts):
    for w in set(text.split()):
        if w in word_to_label_votes:
            word_to_label_votes[w].append(labels[i])

word_labels = []
valid_word_indices = []
valid_words = []
for j, w in enumerate(words):
    votes = word_to_label_votes[w]
    if votes:
        avg_vote = np.mean(votes)
        word_labels.append(1 if avg_vote >= 0.5 else 0)
        valid_word_indices.append(j)
        valid_words.append(w)
    else:
        word_labels.append(1)
        valid_word_indices.append(j)
        valid_words.append(w)

word_labels = np.array(word_labels)
X_words_lda = word_vectors_for_pca[valid_word_indices]

try:
    lda_words = LinearDiscriminantAnalysis(solver='svd')
    lda_words_coords = lda_words.fit_transform(X_words_lda, word_labels)
    lda_words_df = pd.DataFrame({
        "word": valid_words,
        "LD1": lda_words_coords[:, 0],
        "sentiment_label": ['positive' if l==1 else 'negative' for l in word_labels]
    })
    display(lda_words_df.round(4))
    
    fig, ax = plt.subplots(figsize=(11, 6))
    colors_w = {'positive': '#2a9d8f', 'negative': '#e76f51'}
    for sent, group in lda_words_df.groupby('sentiment_label'):
        ax.scatter(group.index, group['LD1'], s=100, label=sent, color=colors_w[sent], edgecolor='black', linewidth=0.6)
    for _, row in lda_words_df.iterrows():
        ax.annotate(row['word'], (lda_words_df.index[lda_words_df['word']==row['word']][0], row['LD1']), 
                    xytext=(0, 10), textcoords="offset points", ha='center', fontsize=9, rotation=45)
    ax.axhline(0, color="gray", linestyle="--", linewidth=0.8, alpha=0.5)
    ax.set(title="LDA: Word Embeddings - Class Separation (Step-wise Visualization)", 
           xlabel="Words", ylabel="Linear Discriminant 1 (LD1)")
    ax.legend()
    ax.grid(alpha=0.25)
    plt.tight_layout()
    plt.show()
except Exception as e:
    print(f"LDA on words note: {e}")

In [ ]:
# LDA on flattened sentence embeddings
try:
    lda_sent = LinearDiscriminantAnalysis(solver='svd')
    lda_sent_coords = lda_sent.fit_transform(flattened_vectors, labels.astype(int))
    lda_sent_df = pd.DataFrame({
        "review_id": review_df["review_id"],
        "sentiment": review_df["sentiment"],
        "LD1": lda_sent_coords[:, 0],
    })
    display(lda_sent_df.round(4))
    
    fig, ax = plt.subplots(figsize=(10, 6))
    colors_s = {'positive': '#2a9d8f', 'negative': '#e76f51'}
    for sent, group in lda_sent_df.groupby('sentiment'):
        ax.scatter(group['review_id'], group['LD1'], s=200, label=sent, color=colors_s[sent], edgecolor='black', linewidth=1)
        for _, row in group.iterrows():
            ax.annotate(f"R{row['review_id']}", (row['review_id'], row['LD1']), xytext=(0, 10), textcoords="offset points", ha='center', fontsize=10)
    ax.axhline(0, color="gray", linestyle="--", linewidth=0.8, alpha=0.5)
    ax.set(title="LDA: Flattened Sentence Embeddings - Supervised Class Separation", 
           xlabel="Review ID", ylabel="Linear Discriminant 1 (LD1)")
    ax.set_xticks(review_df['review_id'])
    ax.legend()
    ax.grid(alpha=0.25)
    plt.tight_layout()
    plt.show()
    
    print("\nLDA provides supervised dimensionality reduction maximizing class separability.")
except Exception as e:
    print(f"LDA on sentences note: {e}")

## 12. Combined PCA+LDA Visualization Pipeline Summary

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# PCA words
axes[0,0].scatter(pca_words_df["PC1"], pca_words_df["PC2"], s=60, color="#277da1", alpha=0.7)
for _, row in pca_words_df.iterrows():
    axes[0,0].annotate(row["word"], (row["PC1"], row["PC2"]), xytext=(3,3), textcoords="offset points", fontsize=8)
axes[0,0].set_title("PCA - Word Embeddings")
axes[0,0].grid(alpha=0.3)

# PCA sentences
for sent, group in pca_sent_df.groupby('sentiment'):
    axes[0,1].scatter(group['PC1'], group['PC2'], s=100, label=sent, color=colors[sent], alpha=0.7)
axes[0,1].set_title("PCA - Sentence Embeddings")
axes[0,1].legend()
axes[0,1].grid(alpha=0.3)

# Training loss
axes[1,0].plot(losses, color='#b23a48')
axes[1,0].set_title("Training Loss (L2 Reg + Balancing)")
axes[1,0].set_xlabel("Epoch")
axes[1,0].grid(alpha=0.3)

# CV results
axes[1,1].bar(['Fold 1', 'Fold 2'], cv_df['acc'], color=['#2a9d8f', '#e76f51'], alpha=0.7)
axes[1,1].set_title("K-Fold CV Accuracy")
axes[1,1].set_ylabel("Accuracy")
axes[1,1].set_ylim([0, 1.1])
axes[1,1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("\nPipeline Summary:")
print("1. Text preprocessing and tokenization ✓")
print("2. Padding sequences ✓")
print("3. Embedding classifier with L2 regularization (weight_decay=0.01) ✓")
print("4. Class balancing via random oversampling ✓")
print("5. Stratified K-Fold Cross-Validation (k=2) ✓")
print("6. PCA on word and sentence embeddings ✓")
print("7. LDA for supervised dimensionality reduction ✓")
print("8. Visualization at each step ✓")